# Chapter 15 — walkthrough

*Computational Methods for Studying Polarization in Arabic Social Media*

This notebook rebuilds **Figure 15.3** and **Figure 15.4** from the data shipped
with the chapter, and ends with exercises.

It needs no tweets. Platform terms allow sharing tweet identifiers, not tweet
text, so what ships instead is the **monthly share of each topic category in
each community** — the table both figures are drawn from. It carries no tweet
text, no account names and no identifiers.

Nothing here is recomputed from the corpus. To do that, see
`build_aggregates.py`. To check every number the chapter states, run
`verify_chapter_numbers.py`.


## Setup

Running locally from inside `chapter15_practices`, the next cell finds the files
by itself. In Google Colab, it asks you to upload `topic_groups.py` and the five
CSV files from `data/`.


In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not os.path.exists("topic_groups.py"):
    print("Upload topic_groups.py and the five CSVs from data/")
    from google.colab import files
    for name in files.upload():
        if name.endswith(".csv"):
            os.makedirs("data", exist_ok=True)
            os.replace(name, os.path.join("data", name))

sys.path.insert(0, ".")
from topic_groups import SELECTED, TOPIC_COLORS, COMMUNITIES, THRESHOLD, TOPIC_GROUPS

DATA = "data"
print(f"{len(TOPIC_GROUPS)} categories in the mapping, {len(SELECTED)} of them plotted")
print(f"communities: {', '.join(COMMUNITIES)}")


## 1. The data

One file per community, plus one for the whole corpus. Each row is a month,
each column a thematic category, and each value that category's **share** of the
month's topic-assigned tweets in that community.


In [ ]:
def load(community):
    df = pd.read_csv(os.path.join(DATA, f"Topics_Discussed_by_{community}.csv"))
    return df.set_index("Year-month")

shares = {c: load(c) for c in COMMUNITIES}
print(f"{shares['Atheists'].shape[0]} months, "
      f"{shares['Atheists'].shape[1]} categories")
shares["Atheists"][SELECTED].head()


### Shares, not counts

Tweet volume changes over the window, so raw counts would mostly show that
change rather than shifts in attention. Two further decisions are built into
these numbers:

- Each tweet is assigned to **one** topic, the highest-probability one. LDA
  actually gives a mixture; keeping only the top topic is what makes the counts
  addable, and it discards the rest.
- The denominator **excludes** the residual `Others` category, which covers 11
  of the 50 topics that did not form a coherent theme.

These files carry 11 of the 18 categories: the nine plotted, plus two the
authors exported while deciding. So a month's columns do not sum to 1 here
either — the remainder sits in categories that were never exported.


In [ ]:
row = shares["Atheists"].iloc[0]
print(f"month: {row.name}")
# The file keeps 11 of the 18 categories, so even all of its columns
# fall short of 1; the rest went to categories the authors did not export.
print(f"all {len(row)} categories in this file : {row.sum():.3f}")
print(f"the nine plotted categories      : {row[SELECTED].sum():.3f}")


## 2. Which categories are shown, and why

A category is retained when the sum of its 42 monthly shares reaches
**1.25** within a community — a mean monthly share of about 3%.

Eight categories clear that bar in all four communities. `Football Clubs` does
not, and is shown anyway. Run the cell and look at why before reading on.


In [ ]:
rows = []
for cat in shares["Atheists"].columns:
    sums = {c: shares[c][cat].sum() for c in COMMUNITIES}
    rows.append({"category": cat, **{c: round(sums[c], 2) for c in COMMUNITIES},
                 "passes": sum(v >= THRESHOLD for v in sums.values()),
                 "plotted": cat in SELECTED})
retention = pd.DataFrame(rows).set_index("category").sort_values(
    ["plotted", "passes"], ascending=False)
retention


`Football Clubs` clears the bar among theists and rationalists only. It is kept
because the contrast between communities is the widest in the data, and that
contrast is itself the finding — attention to football separates these
communities more sharply than attention to religion does.

Two categories that clear the bar in a single community,
`Arguing and Discussing Metaphysics` and `Marriage and Women`, are not plotted.

This is a judgement, not a formula. The chapter says so, and this table is here
so a reader can disagree with it.


## 3. Figure 15.3 — mean monthly share

Each bar is the mean of a category's 42 monthly shares within one community.
The bars of a community sum to roughly 80%, not 100%, because `Others` is
outside the denominator and nine of the eighteen categories are not drawn.


In [ ]:
means = pd.DataFrame({c: shares[c][SELECTED].mean() * 100 for c in COMMUNITIES})
order = means.sum(axis=1).sort_values(ascending=False).index
means = means.loc[order]

ax = means.plot.bar(figsize=(13, 5), width=0.8, rot=0)
ax.set_ylabel("Mean monthly share (%)")
ax.set_xticklabels(range(1, len(order) + 1))
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
for i, cat in enumerate(order, 1):
    print(f"{i} - {cat}")
plt.show()

print("\nbars sum to:")
print(means.sum().round(2).to_string())


## 4. Figure 15.4 — month by month

One panel per community. Each row is a category and each marker is one month.
**Marker size is the monthly share**, not a count of tweets or of accounts.


In [ ]:
months = list(shares["Atheists"].index)
labels = [f"{m.split('_')[0]} Q{(int(m.split('_')[1]) - 1) // 3 + 1}"
          if int(m.split("_")[1]) % 3 == 2 else "" for m in months]

fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True, sharey=True)
for ax, community in zip(axes.ravel(), COMMUNITIES):
    for row, cat in enumerate(SELECTED):
        ax.scatter(range(len(months)), [len(SELECTED) - row] * len(months),
                   s=shares[community][cat] * 4000, color=TOPIC_COLORS[cat],
                   alpha=0.85, edgecolors="white", linewidths=0.5)
    ax.set_title(community, fontsize=11)
    ax.set_yticks([]); ax.set_xticks(range(len(months)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.spines[["top", "right", "left"]].set_visible(False)
fig.legend(handles=[plt.Line2D([], [], marker="o", linestyle="",
                               color=TOPIC_COLORS[c], label=c) for c in SELECTED],
           loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.tight_layout(rect=(0, 0.15, 1, 1))
plt.show()


## 5. Reading a spike

Section 15.5 treats a spike as a place to start reading tweets, not as a result.
Here is the sharpest one in the data.


In [ ]:
suffering = pd.DataFrame({c: shares[c]["Suffering"] * 100 for c in COMMUNITIES})
ax = suffering.plot(figsize=(13, 4))
ax.set_ylabel("Suffering, monthly share (%)")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.show()

peak = suffering.mean(axis=1).idxmax()
print(f"peak month: {peak}")
print(suffering.loc[peak].round(2).to_string())
print(f"\noverall mean: {suffering.mean().mean():.2f}%")


All four communities rise in the same month, and `Suffering` merges two LDA
topics: one about illness, treatment and the coronavirus, one about anger and
misfortune. That makes the pandemic a plausible reading of the rise.

**It is a reading, not a result.** Confirming it means reading tweets from that
month in that category. The data in this notebook cannot settle it, and the
chapter does not claim it does — it says *coinciding with*, not *caused by*.

The same two topics under one label are also why `Suffering` is a loose
category. A different reader could reasonably have kept them apart.


## Exercises

**1. Change the threshold.** Set `THRESHOLD` to 1.0 and to 1.5 and rebuild the
retention table. Which categories enter and leave? Would the chapter's argument
change?

**2. Rank by difference instead of by size.** Figure 15.3 orders categories by
overall prominence. Order them by the gap between the highest and lowest
community instead. Which category comes first, and what does that say about
where these communities actually diverge?

**3. Find another spike.** Pick a category other than `Suffering` and find its
largest month. Does it rise in all four communities or in one? What would you
need in order to explain it, and what could you not conclude from this data
alone?

**4. Argue with the grouping.** Open `topic_groups.py`. Move one topic id to a
different category, rerun the notebook, and see what moves. How much of the
chapter's story survives your edit?


In [ ]:
# Exercise 2 — a starting point
spread = (means.max(axis=1) - means.min(axis=1)).sort_values(ascending=False)
spread.round(2).to_frame("spread between communities (percentage points)")
